# T40 — Demo đầu cuối, và bốn smoke test GPU gom một chỗ

Máy cá nhân không có CUDA, nên T36–T40 mới chỉ kiểm được **hợp đồng** bằng bộ phát hiện giả.
Phiên này nạp mô hình thật **một lần** rồi kiểm cả bốn:

| Ô | Kiểm | Thuộc |
|---|---|---|
| 5 | `HallucinationDetector.from_pretrained` rồi `score` ba ví dụ | T36 |
| 6 | hệ RAG: bốn câu hỏi qua truy xuất → sinh trả lời → chấm | T40 |
| 7 | dịch vụ REST chạy thật trong luồng: `/health` → `ok`, `/score` một mẫu ViHallu có nhãn, `/demo/ask` | T37 |
| 8 | `GET /` trả trang HTML | T39 |

Docker (T38) không chạy được trên Kaggle — vẫn chưa kiểm với GPU, ghi rõ.

Khoảng **10 phút** GPU: một phút nạp, còn lại là sinh trả lời (chậm nhất, greedy 160 token).


In [ ]:
# Ô 1 — lấy code. Chạy lại được nhiều lần.
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/wsunicorn/vihallulens.git"
REPO_DIR = Path("/kaggle/working/vihallulens")


def run(*args, cwd=None):
    print("$", " ".join(str(a) for a in args))
    result = subprocess.run(args, cwd=cwd, capture_output=True, text=True)
    print(result.stdout.strip())
    if result.returncode:
        print(result.stderr.strip())
        raise SystemExit(f"lệnh hỏng: {' '.join(str(a) for a in args)}")
    return result.stdout


if REPO_DIR.exists():
    run("git", "fetch", "--all", cwd=REPO_DIR)
    run("git", "reset", "--hard", "origin/main", cwd=REPO_DIR)
else:
    run("git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR))

os.chdir(REPO_DIR)
run("git", "log", "-1", "--format=%h %s")

In [ ]:
# Ô 2 — cài đặt. Bài học T27 mà lượt đầu của T40 quên: `--no-deps` cố ý không kéo phụ
# thuộc nào về, nên rank-bm25 (BM25 cho hệ RAG) phải cài riêng — Kaggle không có sẵn.
#
# scikit-learn cũng phải nâng: Kaggle có 1.6.1, bundle models/e03_chunk_aware.pkl pickle bằng
# 1.9.0, và pyproject chặn >=1.9,<2 từ T38. Ô 3 nạp thử bundle để chắc trước khi tiêu GPU.
!pip install -q --no-deps -e .
!pip install -q -U bitsandbytes rank-bm25 "scikit-learn>=1.9,<2"

In [ ]:
# Ô 3 — TIỀN KIỂM. Vài giây.
import importlib.util
import sys
from pathlib import Path

sys.path.insert(0, "src")
import torch

from vihallulens.serve.rag import load_corpus

problems = []
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHONG CO"
print(f"  GPU               : {gpu}")
if not torch.cuda.is_available():
    problems.append("khong co CUDA")
for name in ("transformers", "bitsandbytes", "fastapi", "uvicorn", "rank_bm25"):
    ok = importlib.util.find_spec(name) is not None
    print(f"  {name:<18}: {'co' if ok else 'THIEU'}")
    if not ok:
        problems.append(f"thieu {name}")
bundle = Path("models/e03_chunk_aware.pkl")
print(f"  bundle            : {bundle} {'co' if bundle.is_file() else 'THIEU'}")
if not bundle.is_file():
    problems.append("thieu bundle")
else:
    # Nap THU bundle o day. Bundle la pickle cua sklearn; khac phien ban thi co the khong nap
    # duoc, va biet dieu do luc nay re hon biet sau mot phut nap mo hinh 7B.
    import numpy as np
    import sklearn

    from vihallulens import DetectorBundle

    print(f"  sklearn           : {sklearn.__version__}")
    try:
        b = DetectorBundle.load(bundle)
        proba = b.detector.predict_proba(np.zeros((1, b.n_features)))
        print(f"  nap thu bundle    : OK — {b.n_features} cot, tong xac suat {proba.sum():.3f}")
    except Exception as error:
        print(f"  nap thu bundle    : HONG — {type(error).__name__}: {error}")
        problems.append("bundle khong nap duoc tren sklearn nay")
print(f"  kho demo          : {len(load_corpus())} tai lieu")
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
if problems:
    raise SystemExit("TIEN KIEM HONG: " + "; ".join(problems))
print("\nTien kiem dat.")

In [ ]:
# Ô 4 — một mẫu ViHallu có nhãn để gọi /score ở ô 7. Khoảng 1 phút, CPU.
!python scripts/normalize_data.py --dataset vihallu
!python scripts/split_data.py --only vihallu
import pandas as pd

test = pd.read_parquet("data/interim/vihallu_test.parquet")
MAU = test.iloc[0]
print(f"  mau test dau tien: {MAU.sample_id}  nhan {MAU.label}")
print(f"  hoi : {str(MAU.question)[:120]}")
print(f"  dap : {str(MAU.response)[:120]}")

In [ ]:
# Ô 5 — T36: nạp thư viện thật rồi chấm ba ví dụ. Khoảng 1 phút nạp + vài giây.
import json
import time
from pathlib import Path

from vihallulens import HallucinationDetector

t = time.perf_counter()
detector = HallucinationDetector.from_pretrained("models/e03_chunk_aware.pkl", device="cuda")
print(f"  nap xong sau {time.perf_counter() - t:.0f} s: {detector.describe()}")
print(f"  VRAM sau khi nap: {torch.cuda.memory_allocated() / 1e6:,.0f} MB")

KET_QUA = Path("/kaggle/working/ket_qua_t40")
KET_QUA.mkdir(exist_ok=True)

CONTEXT = ("Hà Nội là thủ đô của Việt Nam, nằm bên bờ sông Hồng. Thành phố có lịch sử hơn một "
           "nghìn năm, từng mang tên Thăng Long dưới triều Lý. Dân số nội thành khoảng tám triệu "
           "người. Khí hậu Hà Nội có bốn mùa rõ rệt, mùa đông có thể xuống dưới mười độ.")
VI_DU = [
    ("trung thuc", "Hà Nội từng có tên gọi nào?",
     "Hà Nội từng mang tên Thăng Long dưới triều Lý."),
    ("noi tai", "Hà Nội từng có tên gọi nào?",
     "Hà Nội từng mang tên Thăng Long dưới triều Trần, khi dân số đã lên tới hai mươi triệu "
     "người."),
    ("ngoai lai", "Hà Nội có những công trình nổi tiếng nào?",
     "Hà Nội nổi tiếng với tháp Eiffel thu nhỏ ở hồ Tây và tuyến tàu điện ngầm dài nhất Đông "
     "Nam Á."),
]
ket_qua_t36 = []
for ten, q, r in VI_DU:
    out = detector.score(CONTEXT, q, r)
    ket_qua_t36.append({"vi_du": ten, "question": q, "response": r, **out.to_dict()})
    top = max(out.chunk_attention, key=lambda c: c.share)
    print(f"  [{ten:<10}] nhan {out.label:<10} rui ro {out.risk_score:.3f}  {out.n_chunks} doan, "
          f"nhin nhieu nhat: doan {top.index} ({100 * top.share:.0f} %)  {out.elapsed_ms:.0f} ms")
(KET_QUA / "t36_thu_vien.json").write_text(
    json.dumps(ket_qua_t36, ensure_ascii=False, indent=2), encoding="utf-8")
print("  Ba vi du la de nhin: bo phat hien khong duoc huan luyen tren chung, nhan ra sao ghi vay.")

In [ ]:
# Ô 6 — T40: hệ RAG minh họa, bốn câu hỏi. Khoảng 3–5 phút, phần lớn là sinh trả lời.
from vihallulens.serve.rag import DemoRAG

rag = DemoRAG(detector)
CAU_HOI = ["Đỉnh núi cao nhất Đông Dương là gì?", "Ai chỉ huy chiến dịch Điện Biên Phủ?",
           "Phở xuất hiện ở đâu và khi nào?", "Hồ Hoàn Kiếm gắn với truyền thuyết nào?"]
ket_qua_t40 = []
for q in CAU_HOI:
    out = rag.ask(q)
    ket_qua_t40.append(out.to_dict())
    s = out.score
    print("=" * 78)
    print(f"  HOI    : {q}")
    print("  TAI LIEU: " + " · ".join(d["title"] for d in out.retrieved))
    print(f"  TRA LOI: {out.answer}")
    print(f"  -> nhan {s['label']}, rui ro {s['risk_score']:.3f}, "
          f"sinh {out.elapsed_ms['generate']:.0f} ms, cham {out.elapsed_ms['score']:.0f} ms")
(KET_QUA / "t40_rag.json").write_text(
    json.dumps(ket_qua_t40, ensure_ascii=False, indent=2), encoding="utf-8")

In [ ]:
# Ô 7 — T37: dịch vụ REST chạy THẬT trong một luồng, gọi qua HTTP. Khoảng 1 phút.
import threading

import requests
import uvicorn

from vihallulens.serve.app import create_app

app = create_app(detector=detector)          # dung lai mo hinh da nap, khong nap lan hai
server = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start()

BASE = "http://127.0.0.1:8000"
for _ in range(60):
    try:
        h = requests.get(f"{BASE}/health", timeout=2).json()
        if h["status"] == "ok":
            break
    except requests.RequestException:
        pass
    time.sleep(1)
tom_tat = {k: h[k] for k in ("status", "model_loaded", "reading_model", "vram_allocated_mb")}
print("  /health:", json.dumps(tom_tat, ensure_ascii=False))
assert h["status"] == "ok", h

body = {"context": str(MAU.context), "question": str(MAU.question), "response": str(MAU.response)}
r = requests.post(f"{BASE}/score", json=body, timeout=120)
print(f"  /score  : HTTP {r.status_code}")
score = r.json()
print(f"    mau ViHallu {MAU.sample_id}: nhan that {MAU.label}, du doan {score['label']}, "
      f"rui ro {score['risk_score']:.3f}, {score['n_chunks']} doan")

r2 = requests.post(f"{BASE}/demo/ask", json={"question": "Vịnh Hạ Long thuộc tỉnh nào?"},
                   timeout=300)
print(f"  /demo/ask: HTTP {r2.status_code}")
ask = r2.json()
print(f"    tra loi: {ask['answer'][:140]}")
print(f"    -> nhan {ask['score']['label']}, rui ro {ask['score']['risk_score']:.3f}")

r3 = requests.post(f"{BASE}/score", json={**body, "chunk_strategy": "token_window"}, timeout=10)
print(f"  /score voi chunk_strategy la: HTTP {r3.status_code} (phai la 400)")

(KET_QUA / "t37_rest.json").write_text(json.dumps({
    "health": h,
    "score_request": {**body, "label_that": str(MAU.label), "sample_id": str(MAU.sample_id)},
    "score_response": score, "demo_ask": ask, "bad_strategy_status": r3.status_code,
}, ensure_ascii=False, indent=2), encoding="utf-8")

In [ ]:
# Ô 8 — T39: trang quan sát được phục vụ. Vài giây.
r = requests.get(f"{BASE}/", timeout=10)
html = r.text
print(f"  GET / : HTTP {r.status_code}, {len(html):,} ky tu, "
      f"content-type {r.headers.get('content-type')}")
for marker in ("chunk_attention", "risk_score", "/demo/ask", "char_start"):
    print(f"    co '{marker}': {marker in html}")
(KET_QUA / "t39_trang.html").write_text(html, encoding="utf-8")
print("  Khong co mat nguoi tren Kaggle — trang duoc PHUC VU la dieu kiem duoc o day.")
print("  Nhin bang mat thi mo t39_trang.html tai ve; no goi /health va /score o localhost.")

In [ ]:
# Ô 9 — lấy kết quả về.
for f in sorted(KET_QUA.iterdir()):
    print(f"  {f.name:<24} {f.stat().st_size / 1e3:>7.1f} KB")
print("""
Tai thu muc ket_qua_t40 ve, dan JSON vao PR cua T40. Bon file:
  t36_thu_vien.json   ba vi du qua HallucinationDetector.score
  t40_rag.json        bon cau hoi qua he RAG
  t37_rest.json       /health, /score mot mau co nhan, /demo/ask, va 400 cho chunk_strategy la
  t39_trang.html      trang quan sat nhu server tra ve
""")